In [ ]:
import pandas as pd
import numpy as np
from sklearn import linear_model, metrics
import random
import os
import joblib
import sys
import importlib.util

file_path = os.path.abspath("../app/models.py")
spec = importlib.util.spec_from_file_location("models", file_path)
models = importlib.util.module_from_spec(spec)
sys.modules["models"] = models
spec.loader.exec_module(models)

In [ ]:
SEED = 42 
# Python RNG 
random.seed(SEED) 
# NumPy RNG 
np.random.seed(SEED) 
# Optional: full determinism for sklearn parallel algorithms 
os.environ["PYTHONHASHSEED"] = str(SEED) 
os.environ["OMP_NUM_THREADS"] = "1" 
os.environ["MKL_NUM_THREADS"] = "1"

In [ ]:
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv')
y_test = pd.read_csv('../data/y_test.csv')

In [ ]:
baseline = np.ones(len(y_train))*np.average(y_train)
mse_baseline = metrics.mean_squared_error(y_train, baseline)

clf = linear_model.LinearRegression()
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
mse_st = metrics.mean_squared_error(y_test, y_pred)

print(mse_baseline / mse_st)

5.883119489837891


In [ ]:
class Seeded:
    def __init__(self, estimator):
        self.estimator = estimator

    def __call__(self, *args, **kwargs):
        # Case 1: estimator is a class (e.g., RandomForestRegressor)
        if hasattr(self.estimator, "get_params"):
            # Instantiate a temporary object to inspect parameters
            params = self.estimator().get_params()
            if "random_state" in params and "random_state" not in kwargs:
                kwargs["random_state"] = SEED
            return self.estimator(*args, **kwargs)

        # Case 2: estimator is a function (e.g., train_test_split)
        if hasattr(self.estimator, "__code__"):
            if "random_state" in self.estimator.__code__.co_varnames and "random_state" not in kwargs:
                kwargs["random_state"] = SEED
            return self.estimator(*args, **kwargs)

        # Fallback
        return self.estimator(*args, **kwargs)

In [ ]:
y_pred = models.best_model_regressor_fp(X_train, y_train, X_test, y_test)
mse_best = metrics.mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse_best)
mae  = metrics.mean_absolute_error(y_test, y_pred)
r2   = metrics.r2_score(y_test, y_pred)
mape = metrics.mean_absolute_percentage_error(y_test, y_pred)

print("\n=== Ewaluacja najlepszego modelu (test) ===")
print(f"MSE:   {mse_best:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"MAE:   {mae:.2f}")
print(f"R2:    {r2:.4f}")
print(f"MAPE:  {mape*100:.2f}%")

In [ ]:
### Log-ification of database

y_pred = models.best_model_log_regressor_fp(X_train, y_train, X_test, y_test)

mse_log = metrics.mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse_log)
mae  = metrics.mean_absolute_error(y_test, y_pred)
r2   = metrics.r2_score(y_test, y_pred)
mape = metrics.mean_absolute_percentage_error(y_test, y_pred)

print("\n=== Ewaluacja najlepszego modelu logarytmicznego (test) ===")
print(f"MSE:   {mse_log:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"MAE:   {mae:.2f}")
print(f"R2:    {r2:.4f}")
print(f"MAPE:  {mape*100:.2f}%")

Linear (double split): MSE test (log-trained) = 143242948.8004
Ridge: MSE test (log-trained) = 143234619.0010
Lasso: MSE test (log-trained) = 179006124.8525
kNN: MSE test (log-trained) = 97571409.3408
ElasticNet: MSE test (log-trained) = 248311707.7443
RandomForestRegressor: MSE test (log-trained) = 33185062.4445

Najlepszy model (skala log.): RandomForestRegressor

=== Ewaluacja najlepszego modelu logarytmicznego (test) ===
MSE:   33185062.44
RMSE:  5760.65
MAE:   3614.23
R2:    0.9631
MAPE:  9.97%


In [ ]:
#Gradient Boosting

GB = joblib.load('../app/gb_model.joblib')
y_train_log = np.log1p(y_train.values.ravel())

GB.fit(X_train, y_train_log)
y_pred_log = GB.predict(X_test)
y_pred = np.expm1(y_pred_log)

mse_gb = metrics.mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse_gb)
mae  = metrics.mean_absolute_error(y_test, y_pred)
r2   = metrics.r2_score(y_test, y_pred)
mape = metrics.mean_absolute_percentage_error(y_test, y_pred)

print("\n=== Ewaluacja najlepszego modelu logarytmicznego (test) ===")
print(f"MSE:   {mse_gb:.2f}")
print(f"RMSE:  {rmse:.2f}")
print(f"MAE:   {mae:.2f}")
print(f"R2:    {r2:.4f}")
print(f"MAPE:  {mape*100:.2f}%")



=== Ewaluacja najlepszego modelu logarytmicznego (test) ===
MSE:   83327662.86
RMSE:  9128.40
MAE:   6077.76
R2:    0.9073
MAPE:  15.36%
